No,１

In [ ]:
import os
import shutil
import csv
import re
from pathlib import Path
import json
import tkinter as tk
from tkinter import filedialog

In [17]:
#選択したメインフォルダ内のサブフォルダ名を射出速度に変更


def get_folder_path():
    # ダイアログ用の見えないメインウィンドウを作成
    root = tk.Tk()
    root.withdraw() # 小さいウィンドウを隠す
    root.attributes('-topmost', True) # ダイアログを最前面に表示（VS Codeの裏に隠れるのを防ぐ）

    # フォルダ選択ダイアログを開く
    folder_path = filedialog.askdirectory(title="対象のメインフォルダを選択してください")
    return folder_path

# === ダイアログでメインフォルダのパスを取得 ===
print("ダイアログボックスを開いています...")
main_folder_path = get_folder_path()

# === 新しい名前のリスト ===
new_names = [ "160", "200", "240", "280", "320" ]
# =========================

def rename_subfolders_with_log(target_path_str, names_list):
    if not target_path_str: # ダイアログで「キャンセル」を押した場合
        print("フォルダ選択がキャンセルされました。")
        return

    print(f"選択されたメインフォルダ: {target_path_str}")
    target_dir = Path(target_path_str)
    
    subfolders = sorted([f for f in target_dir.iterdir() if f.is_dir()], key=lambda x: x.name)
    
    if len(subfolders) != len(names_list):
        print(f"エラー: サブフォルダの数（{len(subfolders)}個）と、指定した新しい名前の数（{len(names_list)}個）が一致しません。")
        return

    print("-" * 40)
    
    undo_log = {}

    for folder, new_name in zip(subfolders, names_list):
        new_folder_path = folder.with_name(new_name)
        
        if new_folder_path.exists():
            print(f"スキップ: '{new_name}' は既に存在するため変更を見送りました。")
        else:
            original_name = folder.name
            folder.rename(new_folder_path)
            print(f"変更完了: '{original_name}' -> '{new_name}'")
            undo_log[new_name] = original_name

    if undo_log:
        log_file_path = target_dir / "undo_log.json"
        with open(log_file_path, "w", encoding="utf-8") as f:
            json.dump(undo_log, f, ensure_ascii=False, indent=4)
        print("-" * 40)
        print(f"復元用のログファイルを保存しました: {log_file_path.name}")

# 実行
rename_subfolders_with_log(main_folder_path, new_names)

ダイアログボックスを開いています...
選択されたメインフォルダ: C:/Users/0uh2j/OneDrive/Desktop/実験データ2026/本実験データ/202604-本実験データ_再/1.0mm/230℃
----------------------------------------
変更完了: '2026_0402_190133_699' -> '160'
変更完了: '2026_0402_191429_496' -> '200'
変更完了: '2026_0402_192710_191' -> '240'
変更完了: '2026_0402_193502_544' -> '280'
変更完了: '2026_0402_194547_244' -> '320'
----------------------------------------
復元用のログファイルを保存しました: undo_log.json


上記が失敗してしまったとき用の復元コード

In [ ]:
#名前変更してしまったサブフォルダの復元


# ダイアログでフォルダを選択するための関数
def get_folder_path():
    root = tk.Tk()
    root.withdraw() # メインウィンドウを隠す
    root.attributes('-topmost', True) # ダイアログを最前面に表示
    # 復元用のタイトルに変更
    folder_path = filedialog.askdirectory(title="元に戻したいサブフォルダが入っているメインフォルダを選択してください")
    return folder_path

# 名前変更してしまったサブフォルダの復元

# === ダイアログで復元したいメインフォルダのパスを取得 ===
print("ダイアログボックスを開いています...")
main_folder_path = get_folder_path()

def undo_rename(target_path_str):
    # ダイアログでキャンセルされた場合の処理
    if not target_path_str:
        print("フォルダ選択がキャンセルされました。")
        return

    print(f"選択されたメインフォルダ: {target_path_str}")
    
    cleaned_path_str = target_path_str.strip('\"\'')
    target_dir = Path(cleaned_path_str)
    log_file_path = target_dir / "undo_log.json"

    # ログファイルが存在するかチェック
    if not log_file_path.exists():
        print("エラー: 復元用のログファイル (undo_log.json) が見つかりません。")
        return

    # ログファイルを読み込む
    with open(log_file_path, "r", encoding="utf-8") as f:
        undo_log = json.load(f)

    print("-" * 40)
    
    # 記録をもとに元の名前に戻す
    for current_name, original_name in undo_log.items():
        current_folder_path = target_dir / current_name
        original_folder_path = target_dir / original_name

        if current_folder_path.exists():
            current_folder_path.rename(original_folder_path)
            print(f"復元完了: '{current_name}' -> '{original_name}'")
        else:
            print(f"スキップ: '{current_name}' が見つからないため復元できませんでした。")

# 実行
undo_rename(main_folder_path)